## Quadruped robot vision

This project aims to **bring vision to quadruped robots**.

Since quadropeds have near-ground point of view, the object detector was designed to overcome those obsticals. The choice of having the detection of dogs and humans was made to minic the dog behavor of finding their "race" and "best friends". This should allow robots to know the right way of moving.

To achive this, a YOLO with the COCO and Standrod Dogs dataset were chosen.

**YOLO (You Only Look Once)** was selected as it is a popular, highly efficient, real-time object detection algorithm, which uses a convolutional neural network (CNN) with a signal pass of processing for a given image.
**COCO (Common Objects in Context)** is a large-scale dataset that contains images of many objects with annotations to them.
**Stanford Dogs** is a wide dataset of dogs images of different breads.

However, the datasets do not provide the right training data for the purpose, so a new custom dataset was developed based on those two. They randomly crop parts of the images to simulate low-angle views.

The best training was run for **23.890 hours** on a CPU completing 6 epochs.

In [19]:
# Main imports
import os
import json
import random
import xml.etree.ElementTree as ET # Stanford Dogs XML
import cv2
import numpy as np
from tqdm import tqdm # Nice progress bar
from ultralytics import YOLO
import torch

In [20]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [21]:
# Main paths
COCO_IMAGES = "data/coco2017/train2017"
COCO_ANN = "data/coco2017/annotations/instances_train2017.json"

DOG_ROOT = "data/stanford-dogs-dataset"
DOG_IMG_DIR = DOG_ROOT + "/images/Images"
DOG_ANN_DIR = DOG_ROOT + "/annotations/Annotation"

# Custom dataset, which is balanced for humans and dogs
OUTPUT_ROOT = "dataset_balanced"
IMG_OUT = OUTPUT_ROOT + "/images"
LBL_OUT = OUTPUT_ROOT + "/labels"

for p in [OUTPUT_ROOT, IMG_OUT, LBL_OUT]:
    os.makedirs(p, exist_ok=True)

In [22]:
# Convert (normalize) bounding box coordinates into YOLO's format
def save_yolo_label(label_path, cls, xmin, ymin, xmax, ymax, img_w, img_h):
    x_center = ((xmin + xmax) / 2) / img_w
    y_center = ((ymin + ymax) / 2) / img_h
    w = (xmax - xmin) / img_w
    h = (ymax - ymin) / img_h

    with open(label_path, "w") as f:
        f.write(f"{cls} {x_center} {y_center} {w} {h}\n")

In [23]:
# Dataset readiness check — skip preparation if data already exists
DATASET_READY = False

if os.path.isdir(IMG_OUT) and os.path.isdir(LBL_OUT):
    imgs = sorted([f for f in os.listdir(IMG_OUT) if f.endswith(".jpg")])
    lbls = sorted([f for f in os.listdir(LBL_OUT) if f.endswith(".txt")])

    if len(imgs) > 0 and len(imgs) == len(lbls):
        HUMAN_COUNT = len([f for f in imgs if f.startswith("human")])
        DOG_COUNT = len([f for f in imgs if f.startswith("dog")])
        print(f"Dataset found: {len(imgs)} total samples")
        print(f"  Humans: {HUMAN_COUNT}, Dogs: {DOG_COUNT}")

        # Ensure train/val split files exist
        if not os.path.exists(OUTPUT_ROOT + "/train.txt") or not os.path.exists(OUTPUT_ROOT + "/val.txt"):
            print("Regenerating train/val split...")
            all_imgs = imgs.copy()
            random.shuffle(all_imgs)
            split = int(0.8 * len(all_imgs))
            train_files = all_imgs[:split]
            val_files = all_imgs[split:]

            with open(OUTPUT_ROOT + "/train.txt", "w") as f:
                for img in train_files:
                    f.write(os.path.join(IMG_OUT, img) + "\n")
            with open(OUTPUT_ROOT + "/val.txt", "w") as f:
                for img in val_files:
                    f.write(os.path.join(IMG_OUT, img) + "\n")
            print("Split regenerated.")

        # Ensure dataset.yaml exists
        if not os.path.exists("dataset.yaml"):
            yaml = f"""
path: {os.path.abspath(OUTPUT_ROOT)}
train: train.txt
val: val.txt

names:
  0: human
  1: dog

"""
            with open("dataset.yaml", "w") as f:
                f.write(yaml)
            print("dataset.yaml regenerated.")

        DATASET_READY = True
        print("Dataset is ready — skipping preparation.")
    else:
        print("Dataset folder exists but is incomplete. Running preparation...")
else:
    print("Dataset not found. Running preparation...")

Dataset folder exists but is incomplete. Running preparation...


In [ ]:
if not DATASET_READY:
    print("Processing humans...")

    with open(COCO_ANN, "r") as f:
        coco = json.load(f)

    images_dict = {img["id"]: img for img in coco["images"]}

    count = 0

    for ann in tqdm(coco["annotations"]):
        if ann["category_id"] != 1:  # COCO "person"
            continue

        img_info = images_dict[ann["image_id"]]
        img_path = os.path.join(COCO_IMAGES, img_info["file_name"])

        if not os.path.exists(img_path):
            continue

        x, y, w, h = ann["bbox"]
        x = int(x)
        y = int(y)
        w = int(w)
        h = int(h)

        # Skip degenerate boxes
        if w <= 0 or h <= 0:
            continue

        img_cv = cv2.imread(img_path)
        if img_cv is None:
            continue
        h_img, w_img = img_cv.shape[:2]

        # crop range 40–70% of height
        ratio = random.uniform(0.4, 0.7)
        lower_y = int(y + h * (1 - ratio))

        xmin = x
        ymin = lower_y
        xmax = x + w
        ymax = y + h

        cropped = img_cv[ymin:ymax, xmin:xmax]

        # Skip empty crops (out-of-bounds or zero-size)
        if cropped.size == 0:
            continue

        out_img_name = f"human_{count}.jpg"
        out_lbl_name = f"human_{count}.txt"

        cv2.imwrite(os.path.join(IMG_OUT, out_img_name), cropped)

        # save YOLO label: entire crop is one bounding box
        save_yolo_label(
            os.path.join(LBL_OUT, out_lbl_name),
            cls=0,  # human
            xmin=0,
            ymin=0,
            xmax=(xmax - xmin),
            ymax=(ymax - ymin),
            img_w=(xmax - xmin),
            img_h=(ymax - ymin)
        )

        count += 1

    print("Human samples:", count)
    HUMAN_COUNT = count
else:
    print("Skipping human processing — dataset already exists.")

Processing humans...


In [ ]:
if not DATASET_READY:
    print("Processing dogs...")

    files_list = []
    for root_dir, _, files in os.walk(DOG_ANN_DIR):
        for f in files:
            files_list.append((root_dir, f))

    dog_count = 0

    for root_dir, ann_file in tqdm(files_list):
        xml_path = os.path.join(root_dir, ann_file)
        tree = ET.parse(xml_path)
        root = tree.getroot()

        img_path = xml_path.replace("annotations", "images").replace("Annotation", "Images") + ".jpg"

        if not os.path.exists(img_path):
            continue

        img_cv = cv2.imread(img_path)
        h_img, w_img = img_cv.shape[:2]

        obj = root.find("object")
        box = obj.find("bndbox")

        xmin = int(box.find("xmin").text)
        ymin = int(box.find("ymin").text)
        xmax = int(box.find("xmax").text)
        ymax = int(box.find("ymax").text)

        # Full dog
        dog_crop = img_cv[ymin:ymax, xmin:xmax]

        out_img_name = f"dog_full_{dog_count}.jpg"
        out_lbl_name = f"dog_full_{dog_count}.txt"

        cv2.imwrite(os.path.join(IMG_OUT, out_img_name), dog_crop)

        # YOLO format: full crop
        save_yolo_label(
            os.path.join(LBL_OUT, out_lbl_name),
            cls=1,
            xmin=0,
            ymin=0,
            xmax=(xmax - xmin),
            ymax=(ymax - ymin),
            img_w=(xmax - xmin),
            img_h=(ymax - ymin)
        )

        dog_count += 1

        # Dog's lower part
        crop_ratio = random.uniform(0.6, 0.8)
        new_ymin = int(ymax - (ymax - ymin) * crop_ratio)

        lb_crop = img_cv[new_ymin:ymax, xmin:xmax]

        out_img_name = f"dog_lower_{dog_count}.jpg"
        out_lbl_name = f"dog_lower_{dog_count}.txt"

        cv2.imwrite(os.path.join(IMG_OUT, out_img_name), lb_crop)

        save_yolo_label(
            os.path.join(LBL_OUT, out_lbl_name),
            cls=1,
            xmin=0,
            ymin=0,
            xmax=(xmax - xmin),
            ymax=(ymax - new_ymin),
            img_w=(xmax - xmin),
            img_h=(ymax - new_ymin)
        )

        dog_count += 1

    print("Dog samples:", dog_count)
    DOG_COUNT = dog_count
else:
    print("Skipping dog processing — dataset already exists.")

Skipping dog processing — dataset already exists.


In [ ]:
if not DATASET_READY:
    print("Balancing dataset...")

    target = max(HUMAN_COUNT, DOG_COUNT)

    def simple_augment(img_cv):
        h, w = img_cv.shape[:2]
        xmin, ymin, xmax, ymax = 0, 0, w, h

        if random.random() < 0.7: # Random brightness
            beta = int((random.random() - 0.5) * 100)
            img_cv = cv2.convertScaleAbs(img_cv, alpha=1.0, beta=beta)

        if random.random() < 0.7: # Random contrast
            factor = 0.7 + random.random() * 0.6
            img_cv = cv2.convertScaleAbs(img_cv, alpha=factor, beta=0)

        if random.random() < 0.3: # Random blur
            k = random.uniform(0.3, 1.2)
            ksize = int(max(3, int(k * 3)))
            if ksize % 2 == 0:
                ksize += 1
            img_cv = cv2.GaussianBlur(img_cv, (ksize, ksize), 0)

        if random.random() < 0.6: # Random scaling
            scale = random.uniform(0.8, 1.2)
            new_w, new_h = int(w * scale), int(h * scale)
            resized = cv2.resize(img_cv, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

            if scale >= 1.0:
                # Center-crop back to original size
                left = (new_w - w) // 2
                top = (new_h - h) // 2
                img_cv = resized[top:top + h, left:left + w]
                # bbox stays full frame
            else:
                # Center-pad back to original size
                pad_left = (w - new_w) // 2
                pad_top = (h - new_h) // 2
                padded = np.zeros((h, w, 3), dtype=np.uint8)
                padded[pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized
                img_cv = padded
                xmin = pad_left
                ymin = pad_top
                xmax = pad_left + new_w
                ymax = pad_top + new_h

        return img_cv, (xmin, ymin, xmax, ymax)


    def augment_class(cls_id, prefix):
        src_imgs = [f for f in os.listdir(IMG_OUT) if f.startswith(prefix)]
        src_lbls = [f for f in os.listdir(LBL_OUT) if f.startswith(prefix)]

        img_label_pairs = list(zip(src_imgs, src_lbls))
        count = len(img_label_pairs)

        with tqdm(total=target, initial=count, desc=f"{prefix} augment") as pbar:
            while count < target:
                img_name, lbl_name = random.choice(img_label_pairs)

                img_path = os.path.join(IMG_OUT, img_name)
                label_path = os.path.join(LBL_OUT, lbl_name)

                img_cv = cv2.imread(img_path)
                h, w = img_cv.shape[:2]

                aug_img_cv, (xmin, ymin, xmax, ymax) = simple_augment(img_cv)

                new_img_name = f"{prefix}_aug_{count}.jpg"
                new_lbl_name = f"{prefix}_aug_{count}.txt"
                new_img_path = os.path.join(IMG_OUT, new_img_name)
                new_lbl_path = os.path.join(LBL_OUT, new_lbl_name)

                cv2.imwrite(new_img_path, aug_img_cv)

                save_yolo_label(
                    new_lbl_path,
                    cls=int(cls_id),
                    xmin=xmin,
                    ymin=ymin,
                    xmax=xmax,
                    ymax=ymax,
                    img_w=w,
                    img_h=h
                )

                count += 1
                pbar.update(1)

        return count


    human_final = augment_class(0, "human")
    dog_final = augment_class(1, "dog")

    print("Final counts:", human_final, dog_final)
else:
    print("Skipping balancing — dataset already exists.")

Skipping balancing — dataset already exists.


In [ ]:
if not DATASET_READY:
    print("Creating train/val split...")

    all_imgs = [f for f in os.listdir(IMG_OUT) if f.endswith(".jpg")]
    random.shuffle(all_imgs)

    split = int(0.8 * len(all_imgs))
    train_files = all_imgs[:split]
    val_files = all_imgs[split:]

    # os.mv() and opening and saving files took a lot of time,
    #  so using files with the paths to the dataset images was
    #  an optimization, which is supported by YOLO
    train_list_path = OUTPUT_ROOT + "/train.txt"
    val_list_path = OUTPUT_ROOT + "/val.txt"

    with open(train_list_path, "w") as f:
        for img in train_files:
            f.write(os.path.join(IMG_OUT, img) + "\n")

    with open(val_list_path, "w") as f:
        for img in val_files:
            f.write(os.path.join(IMG_OUT, img) + "\n")

    print("Done.")
else:
    print("Skipping train/val split — dataset already exists.")

Skipping train/val split — dataset already exists.


In [ ]:
if not DATASET_READY:
    # YOLO YAML file
    yaml = f"""
path: {os.path.abspath(OUTPUT_ROOT)}
train: train.txt
val: val.txt

names:
  0: human
  1: dog

"""

    with open("dataset.yaml", "w") as f:
        f.write(yaml)

    print("Created dataset.yaml")
else:
    print("Skipping dataset.yaml — already exists.")

Skipping dataset.yaml — already exists.


In [ ]:
# Dataset Statistics
if not os.path.isdir(IMG_OUT):
    print("Dataset not found. Run preparation first.")
else:
    imgs = sorted([f for f in os.listdir(IMG_OUT) if f.endswith(".jpg")])
    human_count = len([f for f in imgs if f.startswith("human")])
    dog_count = len([f for f in imgs if f.startswith("dog")])
    total = len(imgs)

    print("=" * 40)
    print("DATASET STATISTICS")
    print("=" * 40)
    print(f"Total samples: {total}")
    print(f"  Humans: {human_count} ({human_count/total*100:.1f}%)")
    print(f"  Dogs:   {dog_count} ({dog_count/total*100:.1f}%)")

    if human_count > 0 and dog_count > 0:
        ratio = max(human_count, dog_count) / min(human_count, dog_count)
        print(f"Imbalance ratio: {ratio:.2f}:1")
        if ratio > 2:
            print("  WARNING: Classes are imbalanced (>2:1)")
        else:
            print("  Classes are reasonably balanced")

    # Image size sampling (first 100 images)
    sample = imgs[:100]
    widths, heights = [], []
    for img_name in sample:
        img_path = os.path.join(IMG_OUT, img_name)
        img = cv2.imread(img_path)
        if img is not None:
            h, w = img.shape[:2]
            widths.append(w)
            heights.append(h)

    if widths:
        print(f"\nImage sizes (sampled {len(widths)} images):")
        print(f"  Width:  min={min(widths)}, max={max(widths)}, mean={sum(widths)//len(widths)}")
        print(f"  Height: min={min(heights)}, max={max(heights)}, mean={sum(heights)//len(heights)}")
        small = sum(1 for w, h in zip(widths, heights) if w < 320 or h < 320)
        if small > 0:
            print(f"  {small} images smaller than 320x320 (will be upscaled)")

    # Bounding box statistics
    lbls = sorted([f for f in os.listdir(LBL_OUT) if f.endswith(".txt")])
    areas, aspect_ratios = [], []
    for lbl_name in lbls[:500]:  # Sample 500 labels
        lbl_path = os.path.join(LBL_OUT, lbl_name)
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    _, x, y, w, h = map(float, parts)
                    areas.append(w * h)
                    if h > 0:
                        aspect_ratios.append(w / h)

    if areas:
        print(f"\nBounding boxes (sampled {len(areas)}):")
        small = sum(1 for a in areas if a < (32/320)**2)
        medium = sum(1 for a in areas if (32/320)**2 <= a < (96/320)**2)
        large = sum(1 for a in areas if a >= (96/320)**2)
        print(f"  Small (<32px):    {small} ({small/len(areas)*100:.1f}%)")
        print(f"  Medium (32-96px): {medium} ({medium/len(areas)*100:.1f}%)")
        print(f"  Large (>96px):    {large} ({large/len(areas)*100:.1f}%)")

        extreme = sum(1 for r in aspect_ratios if r > 5 or r < 0.2)
        if extreme > 0:
            print(f"  {extreme} extreme aspect ratios (>5:1 or <1:5)")

    print("=" * 40)

DATASET STATISTICS
Total samples: 5820
  Humans: 5820 (100.0%)
  Dogs:   0 (0.0%)

Image sizes (sampled 100 images):
  Width:  min=5, max=433, mean=71
  Height: min=6, max=304, mean=75
  100 images smaller than 320x320 (will be upscaled)

Bounding boxes (sampled 500):
  Small (<32px):    0 (0.0%)
  Medium (32-96px): 0 (0.0%)
  Large (>96px):    500 (100.0%)


In [ ]:
# Load semi-pretrained model
model = YOLO("yolov8n.pt")

model.train(
    data="dataset.yaml",
    epochs=50,
    imgsz=320,
    batch=8,
    patience=10 # Epochs before stopping
)

100%|██████████| 6.25M/6.25M [00:00<00:00, 11.5MB/s]


KeyboardInterrupt: 

In [ ]:
# Training Summary
import csv

results_path = None
if os.path.isdir("runs/detect"):
    for entry in os.listdir("runs/detect"):
        if entry.startswith("train"):
            candidate = os.path.join("runs/detect", entry, "results.csv")
            if os.path.exists(candidate):
                results_path = candidate

if not results_path or not os.path.exists(results_path):
    print("No training results found. Run training first.")
else:
    print("=" * 40)
    print("TRAINING SUMMARY")
    print("=" * 40)
    print(f"Results file: {results_path}")

    with open(results_path) as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    if not rows:
        print("No data in results file.")
    else:
        # Find best mAP@50
        best_map50 = 0
        best_epoch = 0
        for row in rows:
            val = float(row.get("metrics/mAP50(B)", 0))
            if val > best_map50:
                best_map50 = val
                best_epoch = int(float(row.get("epoch", 0)))

        print(f"Total epochs: {len(rows)}")
        print(f"Best mAP@50: {best_map50:.4f} (epoch {best_epoch})")

        # Last epoch metrics
        last = rows[-1]
        print(f"\nFinal metrics:")
        print(f"  box_loss: {float(last.get('train/box_loss', 0)):.4f}")
        print(f"  cls_loss: {float(last.get('train/cls_loss', 0)):.4f}")
        print(f"  val/box_loss: {float(last.get('val/box_loss', 0)):.4f}")
        print(f"  val/cls_loss: {float(last.get('val/cls_loss', 0)):.4f}")
        print(f"  mAP@50: {float(last.get('metrics/mAP50(B)', 0)):.4f}")
        print(f"  mAP@50-95: {float(last.get('metrics/mAP50-95(B)', 0)):.4f}")

    # Model file info
    model_dir = os.path.dirname(results_path)
    best_pt = os.path.join(model_dir, "weights", "best.pt")
    last_pt = os.path.join(model_dir, "weights", "last.pt")

    print(f"\nModel files:")
    if os.path.exists(best_pt):
        size_mb = os.path.getsize(best_pt) / (1024 * 1024)
        print(f"  best.pt: {size_mb:.2f} MB")
    if os.path.exists(last_pt):
        size_mb = os.path.getsize(last_pt) / (1024 * 1024)
        print(f"  last.pt: {size_mb:.2f} MB")

    print("=" * 40)

In [ ]:
# Validate
with open("dataset_balanced/val.txt") as f:
    val_files = [line.strip() for line in f]

example_path = random.choice(val_files)

print("Testing on:", example_path)

results = model.predict(source=example_path, imgsz=320, save=True)

Testing on: dataset_balanced/images/dog_full_7240.jpg

image 1/1 /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/dataset_balanced/images/dog_full_7240.jpg: 640x608 1 dog, 56.9ms
Speed: 2.5ms preprocess, 56.9ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 608)
Results saved to /opt/homebrew/runs/detect/train54


In [ ]:
# Auto-detect the latest training run
runs_dir = "/opt/homebrew/runs/detect"
model_paths = []

if os.path.isdir(runs_dir):
    for entry in os.listdir(runs_dir):
        if entry.startswith("train"):
            model_path = os.path.join(runs_dir, entry, "weights", "best.pt")
            if os.path.exists(model_path):
                model_paths.append(model_path)

if not model_paths:
    raise FileNotFoundError("No trained model found in runs/detect/train*/weights/best.pt")

# Sort by modification time (most recent last)
latest_model = max(model_paths, key=os.path.getmtime)
print("Loading model:", latest_model)

model = YOLO(latest_model)

test_image = "/Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/dog.jpg"

print("Testing on:", test_image)

results = model.predict(source=test_image, imgsz=320, save=True)

Testing on: /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/dog.jpg

image 1/1 /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/dog.jpg: 640x640 1 dog, 78.2ms
Speed: 11.0ms preprocess, 78.2ms inference, 10.3ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /opt/homebrew/runs/detect/predict


In [ ]:
# Inference Speed Benchmark
import time

if 'model' not in globals():
    print("No model loaded. Run inference cell first.")
else:
    print("=" * 40)
    print("INFERENCE SPEED")
    print("=" * 40)

    # Use a validation image for benchmarking
    if os.path.exists("dataset_balanced/val.txt"):
        with open("dataset_balanced/val.txt") as f:
            val_files = [line.strip() for line in f if line.strip()]
        bench_image = random.choice(val_files) if val_files else None
    else:
        bench_image = None

    if bench_image and os.path.exists(bench_image):
        # Warm-up
        for _ in range(3):
            model.predict(source=bench_image, imgsz=320, verbose=False)

        # Benchmark
        times = []
        for _ in range(10):
            t0 = time.time()
            model.predict(source=bench_image, imgsz=320, verbose=False)
            t1 = time.time()
            times.append(t1 - t0)

        avg_time = sum(times) / len(times)
        fps = 1.0 / avg_time

        print(f"Image: {os.path.basename(bench_image)}")
        print(f"Image size: 320x320")
        print(f"Avg inference time: {avg_time*1000:.1f} ms")
        print(f"FPS: {fps:.1f}")

        if fps >= 30:
            print("  Real-time capable (>=30 FPS)")
        elif fps >= 15:
            print("  Marginal real-time (15-30 FPS)")
        else:
            print("  Not real-time (<15 FPS)")
    else:
        print("No validation image available for benchmarking.")

    print("=" * 40)

In [ ]:
test_image = "/Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/human lower body.png"

print("Testing on:", test_image)

results = model.predict(source=test_image, imgsz=320, save=True)

Testing on: /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/human lower body.png

image 1/1 /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/human lower body.png: 544x640 1 human, 59.5ms
Speed: 7.7ms preprocess, 59.5ms inference, 0.3ms postprocess per image at shape (1, 3, 544, 640)
Results saved to /opt/homebrew/runs/detect/predict


In [ ]:
test_image = "/Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/dog lower body.jpg"

print("Testing on:", test_image)

results = model.predict(source=test_image, imgsz=320, save=True)

Testing on: /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/dog lower body.jpg

image 1/1 /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/dog lower body.jpg: 512x640 1 dog, 39.6ms
Speed: 2.0ms preprocess, 39.6ms inference, 0.6ms postprocess per image at shape (1, 3, 512, 640)
Results saved to /opt/homebrew/runs/detect/predict


In [ ]:
test_image = "/Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/balls.jpg"

print("Testing on:", test_image)

results = model.predict(source=test_image, imgsz=320, save=True)

Testing on: /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/balls.jpg

image 1/1 /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/balls.jpg: 640x640 1 human, 88.1ms
Speed: 7.7ms preprocess, 88.1ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /opt/homebrew/runs/detect/predict
